In [35]:
"""
End-to-end RAG chain — Retrieval Augmented Generation.

Combines:
  - ChromaDB vector store (via vector_store service)
  - OpenAI embeddings (text-embedding-3-small)
  - LangChain LCEL chain (gpt-5-nano)

Usage:
  rag = RAGChain(tenant_slug="my-tenant")
  result = rag.answer("How do PostgreSQL indexes work?")
  print(result["answer"])
  print(result["sources"])
"""
from json import load
import os
import sys
from dataclasses import dataclass

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

load_dotenv()

sys.path.insert(0, os.path.abspath("../Saas-API/"))

try:
    from app.services.vector_store import ingest_document, query_documents
    VECTOR_STORE_AVAILABLE = True
except ImportError:
    VECTOR_STORE_AVAILABLE = False

print(f"Vector store available: {VECTOR_STORE_AVAILABLE}")

Vector store available: True


In [36]:
# ── Mock retriever for standalone use ────────────────────────────────────────
# Used when running rag_chain.py outside the SaaS API project context
MOCK_KNOWLEDGE_BASE = [
    {
        "text": "PostgreSQL B-tree indexes are the default index type. They support equality (=) and range queries (<, >, <=, >=). Create with: CREATE INDEX idx_name ON table(column).",
        "score": 0.0,  # score filled in at query time
        "metadata": {"title": "PostgreSQL Guide", "document_id": "mock-1", "chunk_index": 0}
    },
    {
        "text": "Composite indexes cover multiple columns. The leftmost column must appear in the WHERE clause. Example: CREATE INDEX idx_user_tenant ON notes(user_id, tenant_id).",
        "score": 0.0,
        "metadata": {"title": "PostgreSQL Guide", "document_id": "mock-1", "chunk_index": 1}
    },
    {
        "text": "Use CREATE INDEX CONCURRENTLY to avoid locking the table during index creation. This takes longer but is safe for production environments with live traffic.",
        "score": 0.0,
        "metadata": {"title": "PostgreSQL Guide", "document_id": "mock-1", "chunk_index": 2}
    },
    {
        "text": "Redis cache-aside pattern: check cache first, on miss query the database, store result in cache with TTL. Always set TTL to prevent stale data and memory exhaustion.",
        "score": 0.0,
        "metadata": {"title": "Redis Guide", "document_id": "mock-2", "chunk_index": 0}
    },
    {
        "text": "JWT access tokens should be short-lived (15-60 minutes). Refresh tokens rotate on each use and should be stored in the database to enable revocation.",
        "score": 0.0,
        "metadata": {"title": "Auth Guide", "document_id": "mock-3", "chunk_index": 0}
    },
]

In [37]:
from openai import embeddings


def mock_query(query: str, top_k: int = 3) -> list[dict]:
    """
    Simple keyword-based mock retriever for standalone testing.
    In production this is replaced by ChromaDB semantic search.
    """
    import numpy as np
    from openai import OpenAI

    client = OpenAI()
    all_texts = [query] + [doc["text"] for doc in MOCK_KNOWLEDGE_BASE]

    response = client.embeddings.create(
        model = "text-embedding-3-small",
        input = all_texts
    )

    embeddings = [item.embedding for item in response.data]

    query_emb = np.array(embeddings[0])

    results = []
    for i, doc in enumerate(MOCK_KNOWLEDGE_BASE):
        doc_emb = np.array(embeddings[i+1])
        score = float(
            np.dot(query_emb, doc_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(doc_emb))
        )
        results.append({**doc, "score": round(score,4)})

    results.sort(key = lambda x: x["score"], reverse=True)
    return results[:top_k]

In [38]:
# ── Data Classes ──────────────────────────────────────────────────────────────
@dataclass
class Source:
    """Represents one retrieved chunk used in generating the answer."""
    document_title: str
    chunk_preview: str
    relevance_score: float 
    document_id: str
    chunk_index: int

@dataclass
class RAGResult:
    """Complete result from a RAG query — answer + provenance."""
    question: str
    answer: str
    sources: list[Source]
    chunks_used: int 
    model: str
    retrieval_scores: list[float]


    def print_formatted(self):
        """Pretty-print the result for CLI use."""
        print(f"\n{'='*65}")
        print(f"Question: {self.question}")
        print(f"{'='*65}")
        print(f"\nAnswer: \n{self.answer}\n")
        print(f"{'-'*65}")
        print(f"Sources used ({self.chunks_used} chunks):")
        for i, source in enumerate(self.sources, 1):
            print(f" [{i}] {source.document_title} "
                  f"(score: {source.relevance_score})"
                  f"chunk #{source.chunk_index}"
                  )
            print(f"      Preview: {source.chunk_preview[:100]}...")
        print(f"\nModel used: {self.model}")
        if self.retrieval_scores:
            print(f"Score range: {min(self.retrieval_scores):.3f} – "
                f"{max(self.retrieval_scores):.3f}")
        else:
            print("Score range: N/A (no chunks retrieved)")
        print(f"{'='*65}\n")

In [ ]:
# ── RAG Chain ─────────────────────────────────────────────────────────────────
class RAGChain:
    """
    End-to-end RAG pipeline for a specific tenant.

    Architecture:
      question → retrieve() → format_context() → LLM → answer

    Separation of concerns:
      retrieve()        : vector search, returns raw chunks
      format_context()  : formats chunks into prompt-ready string
      build_chain()     : LCEL chain that handles prompt + LLM + parsing
      answer()          : orchestrates the full pipeline, returns RAGResult
    """
    SYSTEM_PROMPT = """
You are a helpful assistant that answers questions \
based strictly on the provided context documents.

STRICT RULES:
- Answer ONLY using information present in the context below
- If the context does not contain sufficient information to answer, \
respond with exactly: "I don't have enough information to answer this question."
- Do NOT use any knowledge from your training data
- Do NOT make up facts, statistics, or details not present in the context
- If you quote from the context, use the source label (e.g. [Source 1])
- Be concise and direct — no unnecessary preamble

Context:
{context}"""
    def __init__(
            self,
            tenant_slug: str,
            model: str = "gpt-5-nano",
            min_relevance_score: float = 0.3,
    ):
        """
        tenant_slug: identifies which ChromaDB collection to search
        model: LLM model for generation
        min_relevance_score: chunks below this score are filtered out
                             prevents injecting irrelevant context into prompt
        """
        self.tenant_slug = tenant_slug
        self.model_name = model 
        self.min_relevance_score = min_relevance_score

        self.llm = ChatOpenAI(model = model, temperature=0.0)

        self.chain = self._build_chain()

    
    def _build_chain(self):
        """
        Build the LCEL generation chain.

        Structure: prompt | llm | parser
        The context and question are injected at invoke() time.

        Why LCEL here instead of direct llm.invoke()?
        - Composable — easy to add steps (output parsers, retry logic)
        - Streaming support built in — chain.stream() works automatically
        - Consistent interface — same pattern as all other chains in the project
        """
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", self.SYSTEM_PROMPT),
                ("human", "{question}"),
            ]
        )
        return prompt | self.llm | StrOutputParser()
    
    def retrieve(self, question: str, top_k: int = 5) -> list[dict]:
        """Retrieve top_k most relevant chunks for the question.
        Filters out chunks below min_relevance_score.

        Returns empty list if:
        - Collection is empty (no documents ingested)
        - No chunks meet the minimum relevance threshold
        """
        if VECTOR_STORE_AVAILABLE:
            raw = query_documents(
                tenant_slug=self.tenant_slug,
                query = question,
                top_k = top_k,
            )
        else:
            raw = mock_query(question, top_k=top_k)

        # Filter out low-relevance chunks
        filtered = [ r for r in raw if r["score"] >= self.min_relevance_score]
        return filtered
    
    def format_context(self, chunks: list[dict]) -> str:
        """
        Format retrieved chunks into a context string for the prompt.

        Format:
          [Source 1 - Document Title]:
          chunk text here...

          [Source 2 - Another Document]:
          another chunk here...

        Why label sources?
        - LLM can reference them in its answer ([Source 1] says...)
        - Helps the LLM distinguish between different documents
        - Makes the answer's provenance traceable
        """
        if not chunks:
            return "No relevant information found in the knowledge base."
        parts = []
        for i, chunk in enumerate(chunks, 1):
            title = chunk.get("metadata", {}).get("title", "Documents")
            parts.append(f"[Source {i} - {title}]: \n{chunk['text']}")

        return "\n\n".join(parts)
    
    def answer(self, question:str, top_k: int = 5) -> RAGResult:
        """
        Full RAG pipeline — question in, grounded answer out.

        Steps:
        1. Retrieve relevant chunks from ChromaDB
        2. Filter by min_relevance_score
        3. Format chunks into context string
        4. Run LCEL chain: formatted prompt → LLM → parsed answer
        5. Package result with sources for citation
        """

        chunks = self.retrieve(question, top_k=top_k)
        context = self.format_context(chunks)
        generated_answer = self.chain.invoke({"question": question, "context": context})

        sources = []
        for chunk in chunks:
            meta = chunk.get("metadata", {})
            sources.append(
                Source(
                    document_title = meta.get("title", "Unknown Document"),
                    chunk_preview = chunk["text"][:200],
                    relevance_score = chunk["score"],
                    document_id = meta.get("document_id", "N/A"),
                    chunk_index = meta.get("chunk_index", -1),
                )
            )
        return RAGResult(
            question = question,
            answer = generated_answer,
            sources = sources,
            chunks_used = len(chunks),
            model = self.model_name,
            retrieval_scores = [chunk["score"] for chunk in chunks],
        )
    

    def stream_answer(self, question:str, top_k: int = 5):
        """
        Stream the answer token-by-token as it's generated by the LLM.

        This method is for CLI use where we want to show the answer as it comes in.
        The final RAGResult is not returned here since we don't want to wait until the end.
        """

        chunks = self.retrieve(question, top_k=top_k)
        context = self.format_context(chunks)

        print(f"[Streaming answer using {len(chunks)} chunks]\n")
        for token in self.chain.stream({"question": question, "context": context}):
            yield token

In [50]:
# ── Main Demo ─────────────────────────────────────────────────────────────────
def main():
    print("=" * 65)
    print("RAG CHAIN — END-TO-END DEMONSTRATION")
    print("=" * 65)

    TENANT = "demo-tenant"
    rag = RAGChain(tenant_slug=TENANT, model="gpt-5-nano")

    if VECTOR_STORE_AVAILABLE:
        print("\nIngesting sample documents into ChromaDB...")
        sample_docs = [
            {
                "id": "backend-guide-1",
                "title": "PostgreSQL Indexing Guide",
                "text": """PostgreSQL B-tree indexes are the default index type and support
                equality and range queries. Composite indexes cover multiple columns and
                require the leftmost column to appear in the WHERE clause. Use CREATE INDEX
                CONCURRENTLY to avoid locking the table in production. EXPLAIN ANALYZE
                shows whether your query uses an index scan or a sequential scan.
                Partial indexes only index rows matching a condition — smaller and faster
                than full indexes for subset queries."""
            },
            {
                "id": "backend-guide-2",
                "title": "Redis Caching Patterns",
                "text": """Redis implements the cache-aside pattern where the application
                checks cache first, queries the database on a miss, and stores the result
                with a TTL. Always set TTL on cached keys to prevent memory exhaustion.
                Write-through caching writes to both cache and database simultaneously
                keeping them consistent. Redis sorted sets are ideal for leaderboards
                and rate limiting with sliding window counters."""
            },
            {
                "id": "backend-guide-3",
                "title": "JWT Authentication Guide",
                "text": """JWT access tokens should be short-lived between 15 and 60 minutes.
                Refresh tokens rotate on each use and must be stored in the database
                to enable revocation. Never store sensitive data in the JWT payload
                because it is base64 encoded not encrypted. The token type claim
                prevents access tokens from being used as refresh tokens."""
            },
        ]

        for doc in sample_docs:
            result = ingest_document(
                tenant_slug=TENANT,
                document_id=doc["id"],
                text=doc["text"],
                metadata = {"title": doc["title"]},
                chunk_strategy = "token",
            )
            print(f"  Ingested '{doc['title']}': {result['chunks_stored']} chunks")

    # ── Test Questions ────────────────────────────────────────────────────────
    test_questions = [
        "How do I create a database index without locking the table?",
        "What caching pattern should I use to reduce database load?",
        "How long should JWT access tokens be valid?",
        "What is the capital of France?",   # out-of-context — should trigger fallback
    ]

    print("\n" + "=" * 65)
    print("ANSWERING QUESTIONS")
    print("=" * 65)

    for question in test_questions:
        result = rag.answer(question, top_k=3)
        result.print_formatted()
    
    
    # ── Streaming Demo ────────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("STREAMING DEMO")
    print("=" * 65)
    print("\nQ: What are the best practices for PostgreSQL indexing?")
    print("A: ", end="")
    for token in rag.stream_answer(
        "What are the best practices for PostgreSQL indexing?",
        top_k=3
    ):
        print(token, end="", flush=True)
    print("\n")


if __name__ == "__main__":
    main()



RAG CHAIN — END-TO-END DEMONSTRATION

Ingesting sample documents into ChromaDB...
  Ingested 'PostgreSQL Indexing Guide': 1 chunks
  Ingested 'Redis Caching Patterns': 1 chunks
  Ingested 'JWT Authentication Guide': 1 chunks

ANSWERING QUESTIONS
  [DEBUG] Raw scores: [0.0, 0.0, 0.0]

Question: How do I create a database index without locking the table?

Answer: 
I don't have enough information to answer this question.

-----------------------------------------------------------------
Sources used (0 chunks):

Model used: gpt-5-nano
Score range: N/A (no chunks retrieved)

  [DEBUG] Raw scores: [0.0209, 0.0, 0.0]

Question: What caching pattern should I use to reduce database load?

Answer: 
I don't have enough information to answer this question.

-----------------------------------------------------------------
Sources used (0 chunks):

Model used: gpt-5-nano
Score range: N/A (no chunks retrieved)

  [DEBUG] Raw scores: [0.5075, 0.0, 0.0]

Question: How long should JWT access tokens be